**Utilities:**

In [ ]:
def corpusToIndices(corpus):
    wordIndexDict = {}
    newCorpus = []
    index = 0
    for text in corpus:
        indexed = []
        for word in text.split():
            if not word in wordIndexDict.keys():
                wordIndexDict[word] = index
                index += 1
            indexed.append(wordIndexDict[word])
        newCorpus.append(indexed)
    return wordIndexDict,newCorpus

def applyDict(newText,dictionary):
    indexText = []
    for word in newText.split():
        indexText.append(dictionary[word])
    return indexText

**LSTM: Text classification**

Make sure to understand what a specific module does, what input it takes and what output it produces: https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html

In [ ]:
import torch
import torch.nn as nn

class Textclassifier(nn.Module):
    def __init__(self,vocabSize,dimension):
        super().__init__()
        self.dimension = dimension  #how many dimensions should the embeddings have
        self.vocabSize = vocabSize  #number of tokens in the embedding layer
        self.lstm = nn.LSTM(self.dimension,self.dimension,batch_first=True) #lstm init, input dimensions should be equal to word dimensions
        self.embedding = nn.Embedding(self.vocabSize,self.dimension) #embedding init
        self.classification = nn.Linear(self.dimension,2) #classification layer -> input equals word dimension, output = number of classes

    def forward(self,ins):
        embeds = self.embedding(ins) #retrieve the word embeddings
        output, (h_n, c_n) = self.lstm(embeds) #put them into the lstm
        return self.classification(h_n[0]) #retrieve the last output;


In [ ]:
torch.manual_seed(511) #make sure we get the same results for every run
corpus = ["this exercise is great","i am sad that this exercise will be over soon", "we love doing homework", "this homework was too hard"] #inputs
targets = torch.LongTensor([1,0,1,0]) #targets

indexdict, newcorpus = corpusToIndices(corpus) #transform to numbers (no preprocessing)
print (newcorpus)
print (indexdict)

model = Textclassifier(len(indexdict.keys()),8) #model init -> word vecs in 8 dimension
lossFct = nn.CrossEntropyLoss() #loss init
optimizer = torch.optim.AdamW(model.parameters(),lr=0.005) #optim init

for _ in range(15): #training loop for 15 epochs
  lossAbs = 0
  for text, target in zip(newcorpus,targets): #iterate over data and targets
    inputs = torch.IntTensor([text]) #transform inputs to tensor
    outs = model(inputs) #feed inputs through model
    loss = lossFct(outs,target.unsqueeze(-1)) #calculate loss
    loss.backward() #backward
    optimizer.step() #update
    lossAbs += loss
    optimizer.zero_grad()
  print (lossAbs/len(newcorpus))

posSentence = "i love this exercise"
negSentence = "this exercise is too hard"

posIndex = applyDict(posSentence,indexdict)
negIndex = applyDict(negSentence,indexdict)
print ("\nClassification Results:\nExpected 1 and 0")
print (model(torch.IntTensor([posIndex])))
print (model(torch.IntTensor([negIndex])))

[[0, 1, 2, 3], [4, 5, 6, 7, 0, 1, 8, 9, 10, 11], [12, 13, 14, 15], [0, 15, 16, 17, 18]]
{'this': 0, 'exercise': 1, 'is': 2, 'great': 3, 'i': 4, 'am': 5, 'sad': 6, 'that': 7, 'will': 8, 'be': 9, 'over': 10, 'soon': 11, 'we': 12, 'love': 13, 'doing': 14, 'homework': 15, 'was': 16, 'too': 17, 'hard': 18}
tensor(0.7228, grad_fn=<DivBackward0>)
tensor(0.6845, grad_fn=<DivBackward0>)
tensor(0.6543, grad_fn=<DivBackward0>)
tensor(0.6250, grad_fn=<DivBackward0>)
tensor(0.5951, grad_fn=<DivBackward0>)
tensor(0.5632, grad_fn=<DivBackward0>)
tensor(0.5285, grad_fn=<DivBackward0>)
tensor(0.4900, grad_fn=<DivBackward0>)
tensor(0.4476, grad_fn=<DivBackward0>)
tensor(0.4015, grad_fn=<DivBackward0>)
tensor(0.3532, grad_fn=<DivBackward0>)
tensor(0.3043, grad_fn=<DivBackward0>)
tensor(0.2571, grad_fn=<DivBackward0>)
tensor(0.2134, grad_fn=<DivBackward0>)
tensor(0.1747, grad_fn=<DivBackward0>)

Classification Results:
Expected 1 and 0
tensor([[-0.5910,  0.7298]], grad_fn=<AddmmBackward0>)
tensor([[ 0.824

**LSTM: Text Classification (Batch processing)**

Processing one sample at a time is very inefficient. To make use of the Parallelization capabilities of GPUs, samples are commonly processed in batches.

The issue with this is that sequences (in NLP: sentences) often have different lenghts, but in order to parellelize the computation, the lenghts need to be equal.

To tackle this problem, a technique called *padding* is used:

In [ ]:
posSentence = "i love this exercise"
negSentence = "this exercise is too hard"

posIndex = applyDict(posSentence, indexdict)
negIndex = applyDict(negSentence, indexdict)

# Convert to tensors
pos_tensor = torch.IntTensor(posIndex)
neg_tensor = torch.IntTensor(negIndex)

# Create a list of tensors
batched_sentences = [pos_tensor, neg_tensor]

# Create a nested tensor
nested_input = torch.nested.nested_tensor(batched_sentences)

padded = torch.nested.to_padded_tensor(nested_input, padding=19) #first appearance of the padding index 19 (the index could be anything, preferable it would be the number 0)

print (padded)

tensor([[ 4, 13,  0,  1, 19],
        [ 0,  1,  2, 17, 18]], dtype=torch.int32)


**Adapting the model:**

Only one change needs to be done to the model definition: An additional argument *padding_idx* needs to be passed to the Embedding Layer initialisation!

In [ ]:
import torch
import torch.nn as nn

class Textclassifier(nn.Module):
    def __init__(self,vocabSize,dimension):
        super().__init__()
        self.dimension = dimension  #how many dimensions should the embeddings have
        self.vocabSize = vocabSize  #number of tokens in the embedding layer
        self.lstm = nn.LSTM(self.dimension,self.dimension,batch_first=True) #lstm init, input dimensions should be equal to word dimensions
        self.embedding = nn.Embedding(self.vocabSize+1,self.dimension,padding_idx=19) #embedding init; specify padding ID !!!
        self.classification = nn.Linear(self.dimension,2) #classification layer -> input equals word dimension, output = number of classes

    def forward(self,ins):
        embeds = self.embedding(ins) #retrieve the word embeddings
        output, (h_n, c_n) = self.lstm(embeds) #put them into the lstm
        return self.classification(h_n[0]) #retrieve the last output;


**Datasets and Dataloaders**



In [ ]:
from torch.utils.data import Dataset
class SentimentDataset(Dataset):
  def __init__(self,sentences,targets):
    self.sentences = sentences #we assume that the sentences are already transformed to numbers
    self.targets = targets

  def __getitem__(self,index):
    return torch.LongTensor(self.sentences[index]),self.targets[index] #

  def __len__(self):
    return len(self.sentences)

def collate(batch): #per default, the batch passed to collate is simply a list of objects returned by the __getitem__ method
  sentences, targets = zip(*batch) #unpacking the sentence and target tuples to two separate lists
  nested_input = torch.nested.nested_tensor(list(sentences))
  padded = torch.nested.to_padded_tensor(nested_input, padding=19)

  return padded,torch.LongTensor(targets)


In [ ]:
from torch.utils.data import DataLoader

corpus = ["this exercise is great","i am sad that this exercise will be over soon", "we love doing homework", "this homework was too hard"] #inputs
targets = torch.LongTensor([1,0,1,0]) #targets
indexdict, newcorpus = corpusToIndices(corpus) #transform to numbers (no preprocessing)

dataset = SentimentDataset(newcorpus,targets)
loader = DataLoader(dataset,4,collate_fn=collate)
for sample in enumerate(loader):
  print (sample[1][0])

tensor([[ 0,  1,  2,  3, 19, 19, 19, 19, 19, 19],
        [ 4,  5,  6,  7,  0,  1,  8,  9, 10, 11],
        [12, 13, 14, 15, 19, 19, 19, 19, 19, 19],
        [ 0, 15, 16, 17, 18, 19, 19, 19, 19, 19]])


Note that the batch size affects the optimization steps; with a different batch size, it might be necessary to adapt the learning rate!

In [ ]:
torch.manual_seed(511) #make sure we get the same results for every run

model = Textclassifier(len(indexdict.keys())+1,8) #model init -> word vecs in 8 dimension
lossFct = nn.CrossEntropyLoss(ignore_index=19) #loss init
optimizer = torch.optim.AdamW(model.parameters(),lr=0.005) #optim init => adapt to different value, like 0.01

for _ in range(15): #training loop for 15 epochs
  lossAbs = 0
  nrBatches = 0
  for sample in enumerate(loader): #iterate over data and targets
    inputs = sample[1][0]
    targets = sample[1][1]
    outs = model(inputs) #feed inputs through model
    loss = lossFct(outs,targets) #calculate loss
    loss.backward() #backward
    optimizer.step() #update
    lossAbs += loss
    optimizer.zero_grad()
    nrBatches += 1
  print (lossAbs/nrBatches)

tensor(0.7484, grad_fn=<DivBackward0>)
tensor(0.7394, grad_fn=<DivBackward0>)
tensor(0.7310, grad_fn=<DivBackward0>)
tensor(0.7233, grad_fn=<DivBackward0>)
tensor(0.7160, grad_fn=<DivBackward0>)
tensor(0.7092, grad_fn=<DivBackward0>)
tensor(0.7027, grad_fn=<DivBackward0>)
tensor(0.6965, grad_fn=<DivBackward0>)
tensor(0.6904, grad_fn=<DivBackward0>)
tensor(0.6845, grad_fn=<DivBackward0>)
tensor(0.6786, grad_fn=<DivBackward0>)
tensor(0.6727, grad_fn=<DivBackward0>)
tensor(0.6668, grad_fn=<DivBackward0>)
tensor(0.6607, grad_fn=<DivBackward0>)
tensor(0.6544, grad_fn=<DivBackward0>)


**Single Sample Inference:**

In [ ]:
posIndex = applyDict(posSentence,indexdict)
negIndex = applyDict(negSentence,indexdict)
print ("\nClassification Results:\nExpected 1 and 0")
print (model(torch.IntTensor([posIndex])))
print (model(torch.IntTensor([negIndex])))


Classification Results:
Expected 1 and 0
tensor([[-0.1195,  0.2200]], grad_fn=<AddmmBackward0>)
tensor([[-0.1322, -0.1506]], grad_fn=<AddmmBackward0>)


**Batch Inference:**

**Do not forget padding!**

In [ ]:
batched_sentences = [torch.IntTensor([posIndex]), torch.IntTensor([negIndex])]

# Create a nested tensor
nested_input = torch.nested.nested_tensor(batched_sentences)

padded = torch.nested.to_padded_tensor(nested_input, padding=19) #first appearance of the padding index 19 (the index could be anything, preferable it would be the number 0)
print (padded.squeeze())

print (model(padded.squeeze())) #Notice how the padding token affects the classification result!

tensor([[ 4, 13,  0,  1, 19],
        [ 0,  1,  2, 17, 18]], dtype=torch.int32)
tensor([[-0.0965,  0.1193],
        [-0.1322, -0.1506]], grad_fn=<AddmmBackward0>)


# **LSTM - Sequence Labelling**

In [ ]:
class SequenceLabeller(nn.Module):
    def __init__(self,vocabSize,dimension):
        super().__init__()
        self.dimension = dimension  #how many dimensions should the embeddings have
        self.vocabSize = vocabSize  #number of tokens in the embedding layer
        self.lstm = nn.LSTM(self.dimension,self.dimension,batch_first=True) #lstm init, input dimensions should be equal to word dimensions
        self.embedding = nn.Embedding(self.vocabSize,self.dimension) #embedding init
        self.classification = nn.Linear(self.dimension,2) #classification layer -> input equals word dimension, output = number of classes

    def forward(self,ins):
        embeds = self.embedding(ins) #retrieve the word embeddings
        output, (h_n, c_n) = self.lstm(embeds) #put them into the lstm
        return self.classification(output) #classify all outputs

In [ ]:
corpus2 = ["this exercise is great","i am sad that this exercise will be over soon", "we love doing homework", "this homework was too hard"]
targets = ["nn,noun,nn,nn","nn,nn,nn,nn,nn,noun,nn,nn,nn,nn","nn,nn,nn,noun","nn,noun,nn,nn,nn"]

indexdict, newcorpus = corpusToIndices(corpus) #transform to numbers (no preprocessing)
print (newcorpus)
print (indexdict)

targetDict = {'nn':0,"noun":1}
def targetsToIds(targets,targetDict):
  outer = []
  for target in targets:
    inner = []
    for tag in target.split(","):
      inner.append(targetDict[tag])
    outer.append(inner)
  return outer

print (targetsToIds(targets,targetDict))

targets1 = torch.LongTensor([0, 1, 0, 0])
targets2 = torch.LongTensor([0, 0, 0, 0, 0, 1, 0, 0, 0, 0])
targets3 = torch.LongTensor([0, 0, 0, 1])
targets4 = torch.LongTensor([0, 1, 0, 0, 0])


[[0, 1, 2, 3], [4, 5, 6, 7, 0, 1, 8, 9, 10, 11], [12, 13, 14, 15], [0, 15, 16, 17, 18]]
{'this': 0, 'exercise': 1, 'is': 2, 'great': 3, 'i': 4, 'am': 5, 'sad': 6, 'that': 7, 'will': 8, 'be': 9, 'over': 10, 'soon': 11, 'we': 12, 'love': 13, 'doing': 14, 'homework': 15, 'was': 16, 'too': 17, 'hard': 18}
[[0, 1, 0, 0], [0, 0, 0, 0, 0, 1, 0, 0, 0, 0], [0, 0, 0, 1], [0, 1, 0, 0, 0]]


In [ ]:
torch.manual_seed(511) #make sure we get the same results for every run
corpus = ["this exercise is great","i am sad that this exercise will be over soon", "we love doing homework", "this homework was too hard"] #inputs
targets = [targets1,targets2,targets3,targets4]

indexdict, newcorpus = corpusToIndices(corpus) #transform to numbers (no preprocessing)
print (newcorpus)
print (indexdict)

model = SequenceLabeller(len(indexdict.keys()),8) #model init -> word vecs in 8 dimension
lossFct = nn.CrossEntropyLoss() #loss init
optimizer = torch.optim.AdamW(model.parameters(),lr=0.005) #optim init

for _ in range(15): #training loop for 15 epochs
  lossAbs = 0
  for text, target in zip(newcorpus,targets): #iterate over data and targets
    inputs = torch.IntTensor([text]) #transform inputs to tensor
    outs = model(inputs) #feed inputs through model
    loss = lossFct(outs.squeeze(),target) #calculate loss
    loss.backward() #backward
    optimizer.step() #update
    lossAbs += loss
    optimizer.zero_grad()
  print (lossAbs/len(newcorpus))


posSentence = "i love this exercise"
negSentence = "this exercise is too hard"

posIndex = applyDict(posSentence,indexdict)
negIndex = applyDict(negSentence,indexdict)
print ("\nClassification Results:\nExpected 1 where word == Exercise")
print (model(torch.IntTensor([posIndex])))
print (model(torch.IntTensor([negIndex])))

[[0, 1, 2, 3], [4, 5, 6, 7, 0, 1, 8, 9, 10, 11], [12, 13, 14, 15], [0, 15, 16, 17, 18]]
{'this': 0, 'exercise': 1, 'is': 2, 'great': 3, 'i': 4, 'am': 5, 'sad': 6, 'that': 7, 'will': 8, 'be': 9, 'over': 10, 'soon': 11, 'we': 12, 'love': 13, 'doing': 14, 'homework': 15, 'was': 16, 'too': 17, 'hard': 18}
tensor(0.7888, grad_fn=<DivBackward0>)
tensor(0.7353, grad_fn=<DivBackward0>)
tensor(0.6887, grad_fn=<DivBackward0>)
tensor(0.6457, grad_fn=<DivBackward0>)
tensor(0.6055, grad_fn=<DivBackward0>)
tensor(0.5688, grad_fn=<DivBackward0>)
tensor(0.5363, grad_fn=<DivBackward0>)
tensor(0.5088, grad_fn=<DivBackward0>)
tensor(0.4860, grad_fn=<DivBackward0>)
tensor(0.4664, grad_fn=<DivBackward0>)
tensor(0.4476, grad_fn=<DivBackward0>)
tensor(0.4276, grad_fn=<DivBackward0>)
tensor(0.4058, grad_fn=<DivBackward0>)
tensor(0.3829, grad_fn=<DivBackward0>)
tensor(0.3594, grad_fn=<DivBackward0>)

Classification Results:
Expected 1 where word == Exercise
tensor([[[ 0.6972, -0.8613],
         [ 0.8260, -0.63